# Step 04: Data Relationships & Entity Join Analysis

## Overview
This notebook establishes entity confirmation and cardinality checks across all five processed datasets to ensure **no blind joins**:
1. `employee_attrition_processed.csv` $\leftrightarrow$ `engagement_processed.csv`
2. `employee_attrition_processed.csv` $\leftrightarrow$ `occupation_master.csv`
3. `occupation_master.csv` $\leftrightarrow$ `essential_skills_processed.csv`
4. `occupation_master.csv` $\leftrightarrow$ `software_skills_processed.csv`


In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join("..", "data", "processed")

attr_df = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_attrition_processed.csv"))
perf_df = pd.read_csv(os.path.join(PROCESSED_DIR, "engagement_processed.csv"))
occ_df = pd.read_csv(os.path.join(PROCESSED_DIR, "occupation_master.csv"))
ess_df = pd.read_csv(os.path.join(PROCESSED_DIR, "essential_skills_processed.csv"))
soft_df = pd.read_csv(os.path.join(PROCESSED_DIR, "software_skills_processed.csv"))


---
## 1. Attrition $\leftrightarrow$ Engagement Relationship
- **Table Pair**: `employee_attrition_processed.csv` & `engagement_processed.csv`
- **Join Key**: `EmployeeNumber` (Attrition) == `Employee ID` (Engagement)
- **Entity Confirmation**: Both keys identify individual employees.
- **Cardinality**: 1:1 (for matching IDs)


In [2]:
att_ids = set(attr_df['EmployeeNumber'])
perf_ids = set(perf_df['Employee ID'])
common_ids = att_ids.intersection(perf_ids)

print(f"Attrition Total Employees: {len(att_ids)}")
print(f"Engagement Total Employees: {len(perf_ids)}")
print(f"Overlapping Employee IDs: {len(common_ids)} ({len(common_ids)/len(att_ids)*100:.1f}% of attrition dataset)")

# Test merge to verify 1:1 cardinality
merged_emp = pd.merge(attr_df, perf_df, left_on='EmployeeNumber', right_on='Employee ID', how='inner')
print(f"Inner Join Result Shape: {merged_emp.shape}")
assert merged_emp['EmployeeNumber'].is_unique, "Merged employee table has non-unique IDs!"
print("✔ Entity & Key Confirmation Passed: 1:1 matching on employee key.")


Attrition Total Employees: 1470
Engagement Total Employees: 2845
Overlapping Employee IDs: 731 (49.7% of attrition dataset)
Inner Join Result Shape: (731, 60)
✔ Entity & Key Confirmation Passed: 1:1 matching on employee key.


---
## 2. HR Job Role $\leftrightarrow$ O*NET Occupation Master
- **Table Pair**: `employee_attrition_processed.csv` & `occupation_master.csv`
- **Join Key**: `JobRole` mapped to `O*NET-SOC Code` via explicit role mapping dictionary
- **Cardinality**: Many:1 (Multiple employees share the same HR Job Role)


In [3]:
# Role mapping dictionary connecting HR JobRoles to representative O*NET-SOC codes
role_to_onet = {
    'Sales Executive': '41-4012.00',          # Sales Representatives, Wholesale & Manufacturing
    'Research Scientist': '19-1029.01',       # Bioinformatics Scientists / Research Scientists
    'Laboratory Technician': '29-2012.00',   # Medical & Clinical Laboratory Technicians
    'Manufacturing Director': '11-1021.00',   # General and Operations Managers
    'Healthcare Representative': '41-4011.00',# Sales Representatives, Technical
    'Manager': '11-1021.00',                  # General & Operations Managers
    'Sales Representative': '41-4012.00',     # Sales Representatives
    'Research Director': '11-9121.00',        # Natural Sciences Managers
    'Human Resources': '13-1071.00'           # Human Resources Specialists
}

attr_df['ONET_SOC_Code'] = attr_df['JobRole'].map(role_to_onet)
missing_mappings = attr_df['ONET_SOC_Code'].isnull().sum()
print(f"Unmapped HR Job Roles count: {missing_mappings}")
assert missing_mappings == 0, "Found unmapped HR job roles!"
print("✔ Role to O*NET SOC Mapping Passed: 100% of employee roles mapped to O*NET SOC Codes.")


Unmapped HR Job Roles count: 0
✔ Role to O*NET SOC Mapping Passed: 100% of employee roles mapped to O*NET SOC Codes.


---
## 3. Occupation Master $\leftrightarrow$ Essential & Software Skills
- **Join Key**: `O*NET-SOC Code`
- **Cardinality**: 1:Many (One SOC Code maps to multiple skill/tool records)


In [4]:
soc_in_occ = set(occ_df['O*NET-SOC Code'])
soc_in_ess = set(ess_df['O*NET-SOC Code'])
soc_in_soft = set(soft_df['O*NET-SOC Code'])

print(f"Total Occupations in Master: {len(soc_in_occ)}")
print(f"Occupations with Essential Skills: {len(soc_in_occ.intersection(soc_in_ess))}")
print(f"Occupations with Software Skills: {len(soc_in_occ.intersection(soc_in_soft))}")


Total Occupations in Master: 1016
Occupations with Essential Skills: 910
Occupations with Software Skills: 923
